In [2]:
import pandas as pd
from xml.dom import minidom
import re

# Reading the Data

In [17]:
platinum_df = pd.read_table('./data/matres/platinum.txt', header=None, sep='\t', names=['docid', 'verb1', 'verb2', 'eiid1', 'eiid2', 'relation'])
platinum_df[['eiid1', 'eiid2']] = 'E' + platinum_df[['eiid1', 'eiid2']].astype(str)

# Fixing a typo in the docid
platinum_df.loc[platinum_df['docid'] == 'nyt_20130321_sarcozy', 'docid'] = 'nyt_20130321_sarkozy'

platinum_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE


In [22]:
def purify_text(text):
    # Strip TIMEX3 attributes
    text = re.sub(r'<TIMEX3\b[^>]*/>', '[TIMEX3/]', text)
    text = re.sub(r'<TIMEX3\b[^>]*>', '[TIMEX3]', text)
    text = re.sub(r'</TIMEX3>', '[/TIMEX3]', text)

    event_stack = []

    # Replace EVENT tags with their eid values, capitalized
    def _replace_event_tag(match):
        tag = match.group(0)

        if tag.startswith('</EVENT'):
            if event_stack:
                return f"[/{event_stack.pop()}]"
            return tag

        eid_match = re.search(r'\beid\s*=\s*"([^"]+)"', tag)
        if eid_match:
            eid = eid_match.group(1)
            eid = eid.upper()
            event_stack.append(eid)
            return f"[{eid}]"
        return tag

    text = re.sub(r'</EVENT>|<EVENT\b[^>]*>', _replace_event_tag, text)

    return text

def split_sentences(text):
    sentences = re.split(r'(?<!\bMr\.)(?<!\bMs\.)(?<!\bMrs\.)(?<=[.!?])\s+', text)
    return sentences

In [27]:
unique_docids = platinum_df['docid'].unique()

doc_texts = {}
for docid in unique_docids:
    filename = f"{docid}.tml"
    text_content = minidom.parse('./data/tempeval/' + filename).getElementsByTagName('TEXT')[0]
    text_content_str = ''.join(node.toxml() for node in text_content.childNodes).strip()
    text_content_str = purify_text(text_content_str)
    sentences = split_sentences(text_content_str)
    doc_texts[docid] = {
        'text': text_content_str,
        'sentences': sentences
    }

docs_df = pd.DataFrame.from_dict(doc_texts, orient='index').reset_index()
docs_df.columns = ['docid', 'text', 'sentences']
docs_df = docs_df.set_index('docid')
docs_df.head()

,text,sentences
docid,,
WSJ_20130322_159,Israeli Prime Minister Benjamin Netanyahu [E1]...,[Israeli Prime Minister Benjamin Netanyahu [E1...
nyt_20130322_strange_computer,Our [TIMEX3]digital[/TIMEX3] age is all about ...,[Our [TIMEX3]digital[/TIMEX3] age is all about...
CNN_20130321_821,Barack Obama would [E1]make[/E1] a great stand...,[Barack Obama would [E1]make[/E1] a great stan...
nyt_20130321_cyprus,A Cyprus [E2001]exit[/E2001] from the euro uni...,[A Cyprus [E2001]exit[/E2001] from the euro un...
bbc_20130322_1353,Israel's prime minister has [E1]apologised[/E1...,[Israel's prime minister has [E1]apologised[/E...


# Creating Context

In [25]:
def create_context_window(sentences, eiid1, eiid2, padding = 1):
    sentence_indices = []
    for i, sentence in enumerate(sentences):
        if f'[{eiid1}]' in sentence or f'[{eiid2}]' in sentence:
            sentence_indices.append(i)

    if not sentence_indices:
        return ""

    start_index = max(0, min(sentence_indices) - padding)
    end_index = min(len(sentences), max(sentence_indices) + padding + 1)

    context_window = ' '.join(sentences[start_index:end_index])
    return context_window

In [28]:
platinum_df['context_window'] = platinum_df.apply(lambda row: create_context_window(docs_df.loc[row['docid'], 'sentences'], row['eiid1'], row['eiid2']), axis=1)
platinum_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation,context_window
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE,Israeli Prime Minister Benjamin Netanyahu [E1]...
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE,Israeli Prime Minister Benjamin Netanyahu [E1]...
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE,Israeli Prime Minister Benjamin Netanyahu [E1]...
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE,Israeli Prime Minister Benjamin Netanyahu [E1]...
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE,Israeli Prime Minister Benjamin Netanyahu [E1]...
